# 01 - Data collection & exploratory analysis
**Endangered Pollinator Identification System** - MSDS688 case study

This notebook builds the dataset behind the system:

1. **IUCN cross-reference table** - Red List category for every species in scope.
2. **Training images** - research-grade, openly licensed iNaturalist photos (one per observation).
3. **Occurrence sample** - year-stratified coordinates used later by the conservation rule engine.
4. **EDA** - class balance, IUCN mix, geography, time coverage.

### Why these species?
The scope mixes **five IUCN-threatened bumble bees** with **visually similar, common species**
(plus honey bee, carpenter bee and monarch). Telling a Critically Endangered *Bombus affinis*
from an abundant *Bombus impatiens* is the hard, useful problem; most unidentified iNaturalist
bee observations are stuck at genus *Bombus* for exactly this reason.

In [ ]:
# --- Environment setup: works on Google Colab and locally -------------------------------
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/MaryCassieOdie/MSDS688-Case-Study.git"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = "/content/MSDS688-Case-Study"
    if not os.path.exists(REPO_DIR):
        # If the repo is private, upload the project folder to Drive instead and set
        # REPO_DIR = "/content/drive/MyDrive/MSDS688-Case-Study"
        subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=False)
    # Data and outputs live on Drive so they survive Colab runtime resets.
    DRIVE_ROOT = "/content/drive/MyDrive/MSDS688-Case-Study"
    os.environ["PROJECT_DATA_DIR"] = f"{DRIVE_ROOT}/data"
    os.environ["PROJECT_OUTPUT_DIR"] = f"{DRIVE_ROOT}/outputs"
else:
    REPO_DIR = os.path.abspath("..")

sys.path.insert(0, os.path.join(REPO_DIR, "src"))

# --- LITE_MODE: quick CPU test of the whole pipeline -------------------------------------
# Lite = 30 images/species at 160 px, 3+2 epochs, small API samples; runs in ~10 min on a
# laptop CPU. Same code paths as the full run, but the numbers are NOT meaningful for the
# write-up. Lite data/outputs go to separate *_lite folders, so they never mix with a full run.
#   None  -> auto: lite when running locally, full on Colab
#   True / False -> force. Restart the kernel after changing it.
LITE_MODE = None
if LITE_MODE is None:
    LITE_MODE = os.environ.get("POLLINATOR_LITE", "0" if IN_COLAB else "1") in ("1", "true", "True")
os.environ["POLLINATOR_LITE"] = "1" if LITE_MODE else "0"
# Optional: official IUCN Red List API token (otherwise iNaturalist's IUCN mirror is used)
# os.environ["IUCN_API_TOKEN"] = "..."   # on Colab, prefer: from google.colab import userdata

from pollinator_id import config
config.ensure_dirs()
print("MODE   ->", "LITE (CPU smoke test - results not meaningful)" if config.LITE else "FULL")
print("data   ->", config.DATA_DIR)
print("output ->", config.OUTPUT_DIR)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pollinator_id import inaturalist, iucn, dataset
from pollinator_id.config import SPECIES, CLASS_NAMES

pd.DataFrame([s.__dict__ for s in SPECIES])

## 1. IUCN Red List cross-reference
Categories are fetched live, not hard-coded. With `IUCN_API_TOKEN` set, the official IUCN Red List
API v4 is used; otherwise the global IUCN assessment mirrored on each iNaturalist taxon record.

In [ ]:
client = inaturalist.INatClient()
iucn_table = iucn.lookup_statuses(CLASS_NAMES, client)
iucn_table.to_csv(config.IUCN_CSV, index=False)
iucn_table[["scientific_name", "common_name", "iucn_code", "iucn_category", "threatened", "source"]]

## 2. Training images
Up to `IMAGES_PER_CLASS` (400; lite: 30) of the most recent research-grade observations per species
(CC0 / CC-BY / CC-BY-NC photos, wild organisms only). Only the **first photo** of each
observation is used so the same individual cannot appear in both train and test.
Re-running is safe: existing metadata and images are reused.

*Expect ~2 minutes of API calls plus a few minutes of image downloads.*

In [ ]:
if config.METADATA_CSV.exists():
    meta = pd.read_csv(config.METADATA_CSV)
    print(f"Loaded existing metadata: {len(meta)} rows")
else:
    frames = []
    for name in CLASS_NAMES:
        df = inaturalist.collect_species_metadata(client, name, n_images=config.IMAGES_PER_CLASS)
        print(f"{name:24s} {len(df):4d} observations")
        frames.append(df)
    meta = pd.concat(frames, ignore_index=True).drop_duplicates("observation_id")
    meta = inaturalist.download_images(meta)
    meta = dataset.assign_splits(meta)
    meta.to_csv(config.METADATA_CSV, index=False)

# Photo credits, required by the CC licences.
meta[["filepath", "url", "license", "attribution"]].to_csv(config.DATA_DIR / "ATTRIBUTION.csv", index=False)
meta.groupby(["label", "split"]).size().unstack()[["train", "val", "test"]]

## 3. Occurrence sample for the conservation rule engine
Up to `OCC_PER_YEAR` research-grade records **per species per year** over the last `OCC_YEARS_BACK` years (coordinates only, no images).
Stratifying by year keeps abundant species from being represented only by recent records,
which the "possible local decline" rule needs. Common species are included on purpose:
they act as a proxy for **survey effort** in each area.

*Full mode: 14 years x 12 species = ~170 API calls, about 3-4 minutes (lite: ~1 minute).*

In [ ]:
from datetime import date
YEARS = range(date.today().year - config.OCC_YEARS_BACK + 1, date.today().year + 1)
if config.OCCURRENCES_CSV.exists():
    occ = pd.read_csv(config.OCCURRENCES_CSV)
else:
    occ = pd.concat([inaturalist.collect_occurrences(client, n, YEARS, per_year=config.OCC_PER_YEAR) for n in CLASS_NAMES],
                    ignore_index=True)
    occ.to_csv(config.OCCURRENCES_CSV, index=False)
print(len(occ), "occurrence records;", f"{occ.coordinates_obscured.mean():.1%} with obscured coordinates")
occ.pivot_table(index="label", columns="year", values="observation_id", aggfunc="count", fill_value=0)

## 4. Exploratory analysis

In [ ]:
STATUS_COLORS = {"CR": "#b2182b", "EN": "#ef8a62", "VU": "#f4a582", "NT": "#fddbc7",
                 "LC": "#92c5de", "DD": "#bababa", "NE": "#e0e0e0"}
codes = iucn_table.set_index("scientific_name")["iucn_code"]

counts = meta["label"].value_counts().reindex(CLASS_NAMES)
fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(counts.index, counts.values, color=[STATUS_COLORS[codes[n]] for n in counts.index], edgecolor="k", lw=.4)
for i, n in enumerate(counts.index):
    ax.text(counts[n] + 3, i, codes[n], va="center", fontsize=8)
ax.invert_yaxis(); ax.set_xlabel("images"); ax.set_title("Images per species (colour = IUCN category)")
fig.tight_layout(); fig.savefig(config.FIGURE_DIR / "class_counts.png", dpi=150)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
for name in CLASS_NAMES:
    sub = meta[meta.label == name]
    thr = codes[name] in iucn.THREATENED
    ax.scatter(sub.longitude, sub.latitude, s=10 if thr else 3, alpha=.8 if thr else .25,
               color=STATUS_COLORS[codes[name]] if thr else "gray",
               label=f"{name} [{codes[name]}]" if thr else None, zorder=3 if thr else 1)
ax.set_xlim(-170, -50); ax.set_ylim(15, 70)
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude"); ax.grid(alpha=.3)
ax.set_title("Training observations - threatened species highlighted, others in gray")
ax.legend(fontsize=8, loc="lower left")
fig.savefig(config.FIGURE_DIR / "training_map.png", dpi=150, bbox_inches="tight")

In [ ]:
meta["year"] = pd.to_datetime(meta.observed_on, errors="coerce").dt.year
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
meta.year.value_counts().sort_index().plot.bar(ax=axes[0], color="#4d9221")
axes[0].set_title("Training images by observation year")
(meta.groupby("label").coordinates_obscured.mean().reindex(CLASS_NAMES) * 100).plot.barh(ax=axes[1], color="#8073ac")
axes[1].set_title("% of observations with obscured coordinates"); axes[1].invert_yaxis()
fig.tight_layout()

In [ ]:
# Sample images - note how similar several bumble bees look.
from PIL import Image
fig, axes = plt.subplots(len(CLASS_NAMES), 5, figsize=(10, 2.1 * len(CLASS_NAMES)))
for i, name in enumerate(CLASS_NAMES):
    for j, fp in enumerate(meta[meta.label == name].sample(5, random_state=1).filepath):
        axes[i, j].imshow(Image.open(config.IMAGE_DIR / fp)); axes[i, j].axis("off")
    axes[i, 0].set_title(f"{name} [{codes[name]}]", fontsize=9, loc="left")
fig.tight_layout(); fig.savefig(config.FIGURE_DIR / "sample_images.png", dpi=110)

### Observations to discuss
* **Class balance** - are threatened species (e.g. *B. occidentalis*) under-represented? Class
  weights in notebook 02 compensate for any imbalance.
* **Geographic bias** - observations cluster near population centres; the rule engine treats
  sparse areas as *under-surveyed* rather than *empty*.
* **Obscured coordinates** - threatened taxa are often auto-obscured, which is why notebook 04
  uses a coarse spatial grid.